# 11b: Leakage Dose Measurement on LUNA16 Cohort (Step A)
Measures the true leakage dose: fraction of test slices whose subject has >= 1 slice in train.
Compares random_image, block_shuffled (equal cut & changepoint), block_contiguous, and true_subject_split across 20 seeds.


In [ ]:
import sys, os, json, time, types, glob, csv
import numpy as np
from collections import defaultdict, Counter

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")

LIB = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")

from iqoth import splits, external
print("Imported iqoth successfully from", LIB)


In [ ]:
LUNA_ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "annotations.csv" in files and not LUNA_ROOT:
            LUNA_ROOT = root
assert LUNA_ROOT, f"LUNA16 dataset not found in /kaggle/input"
print("Found LUNA16 root:", LUNA_ROOT)

nodule_uids = set()
ann_file = os.path.join(LUNA_ROOT, "annotations.csv")
if os.path.exists(ann_file):
    with open(ann_file) as f:
        for row in csv.DictReader(f):
            nodule_uids.add(row["seriesuid"])
print(f"Total nodule-positive UIDs in annotations.csv: {len(nodule_uids)}")

mhd_files = []
for sdir in ("subset0", "subset1"):
    p = os.path.join(LUNA_ROOT, sdir)
    if os.path.isdir(p):
        mhd_files.extend(glob.glob(f"{p}/**/*.mhd", recursive=True))
if not mhd_files:
    mhd_files = glob.glob(f"{LUNA_ROOT}/**/*.mhd", recursive=True)
print(f"Found {len(mhd_files)} .mhd scans in LUNA16")

nodule_scans = [f for f in mhd_files if os.path.splitext(os.path.basename(f))[0] in nodule_uids]
normal_scans = [f for f in mhd_files if os.path.splitext(os.path.basename(f))[0] not in nodule_uids]
cohort = nodule_scans[:15] + normal_scans[:15]
print(f"Selected cohort of {len(cohort)} scans ({len(nodule_scans[:15])} nodule, {len(normal_scans[:15])} normal)")

def get_depth(mhd_path):
    header = {}
    with open(mhd_path, "r", errors="ignore") as f:
        for line in f:
            if "=" in line:
                k, v = line.split("=", 1)
                header[k.strip()] = v.strip()
    dims = [int(x) for x in header.get("DimSize", "512 512 0").split()]
    return dims[2]

cohort_records = []
scan_depths = {}
for mhd in cohort:
    suid = os.path.splitext(os.path.basename(mhd))[0]
    cls = "nodule_cases" if suid in nodule_uids else "normal_cases"
    lbl = 1 if cls == "nodule_cases" else 0
    depth = get_depth(mhd)
    scan_depths[suid] = depth
    for idx in range(depth):
        cohort_records.append({
            "subject_id": suid,
            "class_name": cls,
            "label": lbl,
            "slice_idx": idx,
            "filename": f"{suid}_{idx:04d}.png",
            "pos": len(cohort_records)
        })

print(f"Total cohort records: {len(cohort_records)} across {len(cohort)} subjects")
for cls in ("nodule_cases", "normal_cases"):
    c_recs = [r for r in cohort_records if r["class_name"] == cls]
    subs = {r["subject_id"] for r in c_recs}
    depths = [scan_depths[s] for s in subs]
    print(f"  {cls:<15}: {len(c_recs)} slices, {len(subs)} subjects | depth min={min(depths)}, max={max(depths)}, mean={np.mean(depths):.1f}")


In [ ]:
# Assign equal-cut proxy blocks (15 equal contiguous cuts per class)
records_equal = [dict(r) for r in cohort_records]
for cls in ("nodule_cases", "normal_cases"):
    c_recs = [r for r in records_equal if r["class_name"] == cls]
    c_recs.sort(key=lambda r: r["pos"])
    blocks = np.array_split(np.arange(len(c_recs)), 15)
    for b_idx, idxs in enumerate(blocks):
        for i in idxs:
            c_recs[i]["block_id"] = f"{c_recs[0]['label']}_eq{b_idx}"

# Also assign changepoint proxy blocks if possible, or use equal cuts
# Run leakage dose measurement
SEEDS = list(range(1, 21)) # 20 seeds

results = {}

def measure_dose(recs, strategy, seeds, subject_key="subject_id"):
    doses = []
    sub_straddles = []
    for s in seeds:
        r_copy = [dict(r) for r in recs]
        if strategy == "true_subject_split":
            external.true_subject_split(r_copy, seed=s, subject_key=subject_key)
        else:
            splits.make_split(r_copy, strategy, seed=s)
        
        tr_subs = {r[subject_key] for r in r_copy if r["split"] == "train"}
        te_recs = [r for r in r_copy if r["split"] == "test"]
        
        leaking_slices = sum(1 for r in te_recs if r[subject_key] in tr_subs)
        dose = leaking_slices / len(te_recs) if te_recs else 0.0
        doses.append(dose)
        
        # Sub straddling
        per_sub = defaultdict(set)
        for r in r_copy:
            per_sub[r[subject_key]].add(r["split"])
        straddled = sum(1 for v in per_sub.values() if len(v) > 1)
        sub_straddles.append(straddled / len(per_sub))
    
    return {
        "mean_dose": float(np.mean(doses)),
        "sd_dose": float(np.std(doses)),
        "min_dose": float(np.min(doses)),
        "max_dose": float(np.max(doses)),
        "all_doses": [float(d) for d in doses],
        "mean_sub_straddle": float(np.mean(sub_straddles)),
        "sd_sub_straddle": float(np.std(sub_straddles))
    }

print("=" * 70)
print("LEAKAGE DOSE MEASUREMENT (Fraction of TEST slices with subject in TRAIN)")
print("=" * 70)

for strat in ["random_image", "block_shuffled", "block_contiguous"]:
    res = measure_dose(records_equal, strat, SEEDS if strat != "block_contiguous" else [42])
    results[f"{strat}_equal_blocks"] = res
    print(f"{strat + ' (equal)':<26}: dose = {res['mean_dose']*100:6.2f}% +/- {res['sd_dose']*100:4.2f}%  | sub straddle = {res['mean_sub_straddle']*100:5.1f}%")

# True subject split
res_true = measure_dose(records_equal, "true_subject_split", SEEDS)
results["true_subject_split"] = res_true
print(f"{'true_subject_split':<26}: dose = {res_true['mean_dose']*100:6.2f}% +/- {res_true['sd_dose']*100:4.2f}%  | sub straddle = {res_true['mean_sub_straddle']*100:5.1f}%")

print("\nHALT CONDITION CHECK:")
bs_dose = results["block_shuffled_equal_blocks"]["mean_dose"]
if bs_dose > 0.90:
    print(f"  [HALT TRIGGERED]: block_shuffled dose ({bs_dose*100:.2f}%) is > 90%!")
    halt = True
else:
    print(f"  [CLEARED]: block_shuffled dose ({bs_dose*100:.2f}%) is <= 90%. Lower bound holds.")
    halt = False
results["halt_dose_high"] = halt


In [ ]:
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/leakage_dose.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved results/leakage_dose.json successfully")
